# ElevenLabs Lesson 01: Text-to-Speech Fundamentals

**Text-to-Speech (TTS)** converts written text into natural-sounding audio.
ElevenLabs provides some of the most realistic AI voices available and it all starts with a single API call.

## What you will learn
1. **ElevenLabs SDK** setup and configuration
2. Your **first TTS call** : text in, audio out
3. **Voice Settings** : stability, similarity, speed controls
4. **Saving audio** to MP3 files
5. **Exploring voices** : listing all available voices
6. **Model comparison** : which model to use when

---
> **Prerequisites:** Install the ElevenLabs SDK and get your API key.
> Sign up at [elevenlabs.io](https://elevenlabs.io) to get a free API key.

```bash
# Εγκατάσταση (run in terminal)
pip install elevenlabs python-dotenv

# Δημιουργία .env αρχείου
echo ELEVENLABS_API_KEY=sk_your_key_here > .env
```

In [ ]:
# === Setup: Φόρτωση API key και αρχικοποίηση client ===

import sys
sys.path.insert(0, '.')

from _helpers import setup_client, play_audio, save_audio, timer

client = setup_client()

---
## 1. Your First TTS Call

The simplest ElevenLabs operation: send text, get audio.

Key parameters:
- **`text`** — the text to convert to speech
- **`voice_id`** — which voice to use (we'll use "Adam")
- **`model_id`** — the TTS model (multilingual or turbo)
- **`output_format`** — audio quality (mp3, pcm, etc.)

```
mp3_44100_128
│   │     │
│   │     └── 128 kbps bitrate
│   └──────── 44.1 kHz sample rate
└──────────── MP3 codec
```

In [ ]:
models = client.models.list()

for model in models:
    if model.can_do_text_to_speech:
        print(model.name, "=>", model.model_id)

In [ ]:
voices = client.voices.search(
    page_size=100,
    voice_type="default"
)

for voice in voices.voices:
    print(voice.name, "=>", voice.voice_id)

In [ ]:
# Πρώτη κλήση TTS — Hello World!

with timer("First TTS call"):
    audio = client.text_to_speech.convert(
        text="Hello, welcome to ElevenLabs! This is your first text-to-speech generation.",
        # voice_id="JBFqnCBsd6RMkjVDRZzb",  # Adam # uIZsnBL0YK1S5j69bAih
        voice_id="pNInz6obpgDQGcFmaJgB",  # Bella 
        model_id="eleven_multilingual_v2",
        output_format="mp3_44100_128"
    )

# Αναπαραγωγή μέσα στο notebook
play_audio(audio)

> 💡 **How it works:** `convert()` returns an audio generator. Our `play_audio()` helper
> collects the chunks, saves to a temp file, and uses `IPython.display.Audio` for playback.

---
## 2. Voice Settings

Fine-tune how the voice sounds with `VoiceSettings`:

| Setting | Range | Effect |
|---------|-------|--------|
| **stability** | 0.0 – 1.0 | Low = more expressive/varied, High = more consistent/monotone |
| **similarity_boost** | 0.0 – 1.0 | How closely to match the original voice. Higher = more faithful |
| **style** | 0.0 – 1.0 | Amplifies the voice's speaking style (use sparingly) |
| **speed** | 0.7 – 1.3 | Speaking speed multiplier |
| **use_speaker_boost** | bool | Enhanced clarity (slight latency increase) |

In [ ]:
# Ρύθμιση φωνής - VoiceSettings
from elevenlabs import VoiceSettings

text = "The quick brown fox jumps over the lazy dog. This sentence contains every letter of the alphabet."

# Expressive voice (χαμηλή σταθερότητα = πιο εκφραστικό)
with timer("Expressive TTS"):
    audio_expressive = client.text_to_speech.convert(
        text=text,
        voice_id="JBFqnCBsd6RMkjVDRZzb",
        model_id="eleven_multilingual_v2",
        output_format="mp3_44100_128",
        voice_settings=VoiceSettings(
            stability=0.2,           # Low → more expressive
            similarity_boost=0.8,
            style=0.5,
            speed=1.0,
            use_speaker_boost=True
        )
    )

print("🎭 Expressive voice:")
play_audio(audio_expressive)

In [ ]:
# Stable/monotone voice (υψηλή σταθερότητα = πιο μονότονο)

with timer("Stable TTS"):
    audio_stable = client.text_to_speech.convert(
        text=text,
        voice_id="JBFqnCBsd6RMkjVDRZzb",
        model_id="eleven_multilingual_v2",
        output_format="mp3_44100_128",
        voice_settings=VoiceSettings(
            stability=0.9,           # High → more consistent
            similarity_boost=1.0,
            style=0.0,
            speed=1.0,
            use_speaker_boost=False
        )
    )

print("🤖 Stable voice:")
play_audio(audio_stable)

> 🎧 **Listen to both!** Notice how lower stability sounds more natural and varied,
> while higher stability sounds more robotic but predictable.

---
## 3. Saving Audio to File

Save generated audio as MP3 for later use:

In [ ]:
# Αποθήκευση σε αρχείο MP3

with timer("TTS for file"):
    audio_for_file = client.text_to_speech.convert(
        text="This audio was generated by ElevenLabs and saved as an MP3 file.",
        voice_id="JBFqnCBsd6RMkjVDRZzb",
        model_id="eleven_multilingual_v2",
        output_format="mp3_44100_128",
        voice_settings=VoiceSettings(
            stability=0.5,
            similarity_boost=0.75,
            style=0.0,
            speed=1.0
        )
    )

# Αποθήκευση με τον helper
filepath = save_audio(audio_for_file, "lesson01_hello.mp3")

# Αναπαραγωγή από αρχείο
play_audio(filepath)

---
## 4. Exploring Voices

ElevenLabs offers dozens of pre-made voices. Let's see what's available:

In [ ]:
# Λίστα διαθέσιμων φωνών
from _helpers import show_voices_table

voices = client.voices.get_all()
print(f"📢 Total voices available: {len(voices.voices)}\n")

show_voices_table(client, limit=15)

In [ ]:
# Δοκίμασε μια διαφορετική φωνή!
# Πάρε ένα voice_id από τον πίνακα πάνω και αντικατέστησέ το

VOICE_ID = "Xb7hH8MSUJpSbSDYk0k2"  # Try a different voice from the table above

with timer("Different voice"):
    audio_new_voice = client.text_to_speech.convert(
        text="Every voice has its own character, its own personality. Which one do you prefer?",
        voice_id=VOICE_ID,
        model_id="eleven_multilingual_v2",
        output_format="mp3_44100_128"
    )

play_audio(audio_new_voice)

---
## 5. Model Comparison

ElevenLabs offers several TTS models, each with different tradeoffs:

| Model | Quality | Speed | Languages | Best For |
|-------|---------|-------|-----------|----------|
| `eleven_multilingual_v2` | ⭐⭐⭐⭐⭐ | Moderate | 29 languages | Production, multilingual |
| `eleven_turbo_v2_5` | ⭐⭐⭐⭐ | Fast | English + others | Low-latency applications |
| `eleven_flash_v2` | ⭐⭐⭐ | Very fast | English + others | Real-time, conversational |

Let's compare them side-by-side:

In [ ]:
# Σύγκριση μοντέλων — ίδιο κείμενο, διαφορετικό μοντέλο
import time

comparison_text = "Artificial intelligence is transforming the way we create and consume audio content."

models = [
    ("eleven_multilingual_v2", "Multilingual v2"),
    ("eleven_turbo_v2_5",      "Turbo v2.5"),
    ("eleven_flash_v2",        "Flash v2"),
]

results = []
for model_id, model_name in models:
    try:
        start = time.perf_counter()
        audio = client.text_to_speech.convert(
            text=comparison_text,
            voice_id="JBFqnCBsd6RMkjVDRZzb",
            model_id=model_id,
            output_format="mp3_44100_128"
        )
        # Collect audio to measure timing
        audio_bytes = b"".join(chunk for chunk in audio if chunk)
        ms = (time.perf_counter() - start) * 1000
        
        size_kb = len(audio_bytes) / 1024
        results.append((model_name, ms, size_kb, audio_bytes))
        print(f"✅ {model_name}: {ms:.0f} ms ({size_kb:.1f} KB)")
    except Exception as e:
        print(f"❌ {model_name}: {e}")

# Play the highest quality one
if results:
    print(f"\n🎧 Playing {results[0][0]}:")
    play_audio(results[0][3])

---
## 6. Multilingual TTS

The `eleven_multilingual_v2` model supports **29 languages** — same voice, any language:

In [ ]:
# Πολυγλωσσικό TTS — ίδια φωνή, διαφορετικές γλώσσες!

multilingual_texts = [
    ("English",  "Hello! Welcome to the world of text-to-speech."),
    ("Greek",    "Γεια σας! Καλώς ήρθατε στον κόσμο της σύνθεσης ομιλίας."),
    ("Spanish",  "¡Hola! Bienvenidos al mundo de la síntesis de voz."),
    ("Japanese", "こんにちは！テキスト読み上げの世界へようこそ。"),
]

for lang, text in multilingual_texts:
    with timer(f"{lang} TTS"):
        audio = client.text_to_speech.convert(
            text=text,
            voice_id="JBFqnCBsd6RMkjVDRZzb",
            model_id="eleven_multilingual_v2",
            output_format="mp3_44100_128"
        )
    
    print(f"   🌐 {lang}: \"{text[:50]}...\"")
    save_audio(audio, f"lesson01_{lang.lower()}.mp3")

> ✏️ **Exercise:** Try the following challenges:
> 1. Generate the same sentence with **3 different VoiceSettings** combinations and compare
> 2. Find a voice you like from `show_voices_table()` and use it to generate a poem
> 3. Generate the same text in your native language using `eleven_multilingual_v2`
> 4. Time how long each model takes and note the speed difference

---
## Key Takeaways 📝

| Concept | Detail |
|---------|--------|
| **`text_to_speech.convert()`** | Core function — text in, audio generator out |
| **`VoiceSettings`** | Fine-tune stability, similarity, style, speed |
| **Voice IDs** | Each voice has a unique ID, use `voices.get_all()` to list them |
| **Models** | `multilingual_v2` (best quality), `turbo_v2_5` (fast), `flash_v2` (fastest) |
| **Multilingual** | Same voice speaks 29 languages with `multilingual_v2` |
| **Output formats** | `mp3_44100_128` (high quality), `mp3_22050_32` (compact) |

---
**Next lesson:** Advanced TTS — Streaming, Request Stitching & Building APIs